In [ ]:
# Improved RAG Implementation
# This notebook fixes the issues:
# 1. Retrieves 4-5 chunks instead of 1
# 2. Better chunk size for structured data (course catalogs)
# 3. Improved prompt for formatted responses
# 4. Context validation and relevance filtering

!pip install -q langchain langchain-google-genai faiss-cpu langchain-community PyMuPDF python-docx


In [ ]:
import os
import fitz  # PyMuPDF
from google.colab import files

# LangChain & Vector Store Imports
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Setup API Key
my_api_key = "YOUR_API_KEY_HERE"  # Replace with your actual key
os.environ["GOOGLE_API_KEY"] = my_api_key


In [ ]:
# Extract text from PDF
def extract_from_pdf(file_path):
    """Extracts text from a PDF file."""
    text = ""
    with fitz.open(file_path) as doc:
        for page in doc:
            text += page.get_text() + "\n"
    return text

# Upload and extract
print("--- Upload your course catalog PDF ---")
uploaded = files.upload()

raw_content = ""
for filename in uploaded.keys():
    if filename.endswith('.pdf'):
        raw_content = extract_from_pdf(filename)
        print(f"Extracted {len(raw_content)} characters from {filename}")
    else:
        print(f"Unsupported file: {filename}")


In [ ]:
# IMPROVED CHUNKING STRATEGY
# For course catalogs and structured data:
# - Larger chunk size (1500-2000 chars) preserves complete course entries
# - Overlap of 300-400 chars maintains context between related courses
# - This prevents courses from being split across chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,      # Increased from 1000 - better for structured data
    chunk_overlap=300,    # Increased overlap for better context
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]  # Try to break at paragraphs first
)

docs = text_splitter.create_documents([raw_content])
print(f"✅ Created {len(docs)} chunks")
print(f"Average chunk size: {sum(len(d.page_content) for d in docs) / len(docs):.0f} characters")
print(f"First chunk preview: {docs[0].page_content[:200]}...")


In [ ]:
# Initialize embeddings and vector store
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004",
    google_api_key=my_api_key
)

# Create FAISS vector store
vector_store = FAISS.from_documents(docs, embeddings)
print("✅ Vector store created and indexed!")

# Initialize LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    google_api_key=my_api_key
)


In [ ]:
# IMPROVED PROMPT for better formatting and structured responses
RAG_PROMPT = """You are a helpful college information assistant. Answer the user's question based ONLY on the provided context from college documents.

**Critical Instructions:**
1. Use ONLY information from the provided context. Do not use external knowledge.
2. Format your answer clearly and professionally:
   - Use bullet points (•) or numbered lists when listing multiple items
   - Use **bold text** for important terms, course codes, or subject names
   - Maintain proper structure when presenting course/subject information
   - If listing courses with credits, use a clear format like: "Course Name (Course Code) - X Credits"
3. Preserve structured data formatting:
   - If the context contains tables or structured lists (like course catalogs), maintain that structure
   - Organize information by semester/year if applicable
4. Be concise but comprehensive - include all relevant details
5. If information is missing from context, clearly state: "This information is not available in the provided documents."

**Context from Documents:**
{context}

**User Question:**
{question}

**Your formatted answer:**"""

prompt_template = ChatPromptTemplate.from_template(RAG_PROMPT)
rag_chain = prompt_template | llm | StrOutputParser()

print("✅ RAG chain configured!")


In [ ]:
# IMPROVED RETRIEVAL: Get 4-5 chunks with relevance filtering
def retrieve_and_answer(query, k=5, similarity_threshold=0.7):
    """
    Retrieve multiple relevant chunks and generate answer.
    
    Args:
        query: User's question
        k: Number of chunks to retrieve (default 5)
        similarity_threshold: Minimum similarity score (0.0-1.0)
    """
    
    # 1. Retrieve top-k similar chunks
    similar_docs = vector_store.similarity_search_with_score(query, k=k)
    
    # 2. Filter by similarity threshold (if scores are available)
    # FAISS returns (doc, score) where score is L2 distance (lower = more similar)
    # We'll convert to similarity score (higher = more similar)
    filtered_docs = []
    for doc, score in similar_docs:
        # Convert L2 distance to similarity (approximate)
        # Lower distance = higher similarity
        # Normalize score to 0-1 range (approximation)
        similarity = 1 / (1 + score)  # Simple conversion
        if similarity >= similarity_threshold:
            filtered_docs.append((doc, similarity))
    
    # If filtering removed all docs, use top 3 anyway
    if not filtered_docs:
        print("⚠️ No chunks met similarity threshold, using top 3 chunks")
        filtered_docs = [(doc, 1.0 / (1 + score)) for doc, score in similar_docs[:3]]
    
    # 3. Build context from filtered chunks
    context_text = "\n\n---\n\n".join([
        f"[Chunk {i+1}]\n{doc.page_content}" 
        for i, (doc, sim) in enumerate(filtered_docs[:k])
    ])
    
    print(f"📚 Retrieved {len(filtered_docs)} relevant chunks")
    print(f"📏 Total context length: {len(context_text)} characters")
    
    # 4. Generate answer using RAG chain
    answer = rag_chain.invoke({
        "context": context_text,
        "question": query
    })
    
    return answer, filtered_docs

# Test the improved retrieval
query = "give me name and credit of all subject semester-wise"
print(f"🔍 Question: {query}\n")
print("=" * 60)

answer, used_chunks = retrieve_and_answer(query, k=5, similarity_threshold=0.5)
print("\n💬 Answer:")
print(answer)
print("\n" + "=" * 60)
print(f"\n📊 Used {len(used_chunks)} chunks for this answer")
